# 제5장. 인과추론 기초

## 학습 목표
1. 상관관계와 인과관계의 차이를 명확히 이해하고, 허위 상관과 Simpson's Paradox를 식별할 수 있다
2. 인과 다이어그램(DAG)을 작성하고, 교란변수/매개변수/충돌변수를 구분할 수 있다
3. do-연산자와 반사실적 사고의 개념을 이해하고 기획에 적용할 수 있다
4. 준실험 설계(DID, RDD, IV)의 기본 원리를 이해할 수 있다
5. Python으로 교란변수를 통제하여 인과 효과를 추정할 수 있다

---

### 강의 구조 (3시간)

| 시간 | 파트 | 내용 |
|------|------|------|
| | Part 1 | 상관관계 vs 인과관계 (허위 상관, Simpson's Paradox) + 조사 과제 |
| | Part 2 | 인과 다이어그램 DAG (교란변수, 매개변수, 충돌변수) + 조사 과제 |
| | 휴식 | |
| | Part 3 | 인과 효과 추정 (do-연산자, 반사실, 준실험 설계) + 조사 과제 |
| | Part 4 | 실습: 마케팅 캠페인 효과 시뮬레이션 |
| | Part 5 | 종합 실습: DAG 구축 + 인과 효과 추정 |

---

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly", "scipy"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---

## Part 1. 상관관계 vs 인과관계

---

## 5.1 왜 구분이 중요한가?

### 핵심 차이

| 구분 | 상관관계 (Correlation) | 인과관계 (Causation) |
|------|----------------------|---------------------|
| 정의 | 함께 움직이는 패턴 | 한쪽이 다른 쪽을 **유발** |
| 방향성 | 대칭적 (X ↔ Y) | 비대칭적 (X → Y) |
| 개입 함의 | 없음 | **있음** (X를 바꾸면 Y가 변함) |
| 기획 활용 | 모니터링 (제한적) | 정책 설계 (**핵심**) |
| 확인 방법 | 데이터 분석 | 실험 또는 인과추론 |

### 기획에서의 의미

> 기획의 본질은 **"무엇을 하면 어떤 결과가 나올까?"**에 답하는 것이다.  
> 이 질문에 답하려면 상관관계가 아닌 **인과관계**를 이해해야 한다.

### 상관관계를 인과관계로 오인하면?

1. 효과 없는 정책에 자원을 **낭비**한다
2. 진짜 원인을 놓쳐 문제가 **해결되지 않는다**
3. 의도치 않은 **부작용**이 발생한다

---

In [ ]:
# 허위 상관 시뮬레이션: 아이스크림 판매와 익사 사고
np.random.seed(42)

months = np.arange(1, 13)
# 교란변수: 기온 (여름에 높음)
temperature = 5 + 20 * np.sin((months - 4) * np.pi / 6) + np.random.normal(0, 2, 12)
temperature = np.clip(temperature, -5, 38)

# 아이스크림 판매: 기온에 영향 받음
ice_cream = 50 + 3 * temperature + np.random.normal(0, 5, 12)

# 익사 사고: 기온에 영향 받음 (수영 증가)
drowning = 2 + 0.3 * temperature + np.random.normal(0, 1, 12)
drowning = np.clip(drowning, 0, None)

# 상관계수 계산
corr_ice_drown = np.corrcoef(ice_cream, drowning)[0, 1]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Spurious Correlation (r = {corr_ice_drown:.2f})',
        'True Causal Structure'
    )
)

# 왼쪽: 상관관계 산점도
fig.add_trace(
    go.Scatter(
        x=ice_cream, y=drowning, mode='markers+text',
        marker=dict(size=12, color=temperature, colorscale='RdYlBu_r',
                    colorbar=dict(title='Temp (C)', x=0.45)),
        text=[f'M{m}' for m in months],
        textposition='top center',
        name='Monthly Data'
    ),
    row=1, col=1
)
fig.update_xaxes(title_text='Ice Cream Sales (units)', row=1, col=1)
fig.update_yaxes(title_text='Drowning Incidents', row=1, col=1)

# 오른쪽: DAG 구조
# 노드
dag_nodes = [
    {'x': 2, 'y': 3, 'text': 'Temperature\n(Confounder)', 'color': '#FF6B6B'},
    {'x': 1, 'y': 1, 'text': 'Ice Cream\nSales', 'color': '#4ECDC4'},
    {'x': 3, 'y': 1, 'text': 'Drowning\nIncidents', 'color': '#45B7D1'}
]

for node in dag_nodes:
    fig.add_trace(
        go.Scatter(
            x=[node['x']], y=[node['y']], mode='markers+text',
            marker=dict(size=50, color=node['color']),
            text=[node['text']], textposition='middle center',
            textfont=dict(size=9, color='white'),
            showlegend=False
        ),
        row=1, col=2
    )

# 화살표
for ax, ay, x, y in [(2, 2.7, 1.2, 1.3), (2, 2.7, 2.8, 1.3)]:
    fig.add_annotation(
        x=x, y=y, ax=ax, ay=ay,
        xref='x2', yref='y2', axref='x2', ayref='y2',
        showarrow=True, arrowhead=2, arrowsize=1.5, arrowcolor='gray'
    )

# X 표시 (직접 인과관계 없음)
fig.add_annotation(
    x=2, y=0.8, text='❌ No Direct Cause',
    xref='x2', yref='y2', showarrow=False,
    font=dict(size=11, color='red')
)

fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=2)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=2)

fig.update_layout(height=400, showlegend=False,
                  title_text='Spurious Correlation: Ice Cream Sales vs Drowning')
fig.show()

print(f"💡 아이스크림 판매와 익사 사고의 상관계수: r = {corr_ice_drown:.2f} (강한 정적 상관)")
print("   그러나 아이스크림이 익사를 유발하는 것이 아니다!")
print("   기온(교란변수)이 두 변수 모두에 영향을 미치는 허위 상관")

### 허위 상관 (Spurious Correlation)

두 변수 사이에 직접적인 인과관계가 없음에도 상관관계가 관찰되는 현상.  
대부분 **교란변수(Confounder)**에 의해 발생한다.

```
        기온 (교란변수)
       ↙        ↘
아이스크림 판매    익사 사고
```

### 비즈니스 맥락의 허위 상관

| 관찰된 상관 | 성급한 결론 | 실제 교란변수 |
|------------|------------|------------|
| 마케팅비↑ → 매출↑ | 마케팅 투자가 매출을 높인다 | **성수기** (수요↑ → 마케팅↑ + 매출↑) |
| 교육수준↑ → 성과↑ | 학력이 높으면 성과가 좋다 | **문제해결 능력** |
| 디지털역량↑ → 수익↑ | 디지털 투자하면 수익 증가 | **혁신적 기업문화** 또는 **역인과** |

---

In [ ]:
# Simpson's Paradox 시뮬레이션
np.random.seed(42)

# 가상 데이터: 학과별 합격률
data = {
    'Gender': ['Male'] * 2000 + ['Female'] * 2000,
    'Department': (
        ['Dept A'] * 1000 + ['Dept B'] * 1000 +  # 남성
        ['Dept A'] * 1800 + ['Dept B'] * 200      # 여성 (A에 집중 지원)
    ),
    'Admitted': (
        [1]*300 + [0]*700 +    # 남성 A: 30%
        [1]*600 + [0]*400 +    # 남성 B: 60%
        [1]*630 + [0]*1170 +   # 여성 A: 35%
        [1]*170 + [0]*30       # 여성 B: 85%
    )
}
df_simpson = pd.DataFrame(data)

# 전체 합격률
overall = df_simpson.groupby('Gender')['Admitted'].mean()

# 학과별 합격률
by_dept = df_simpson.groupby(['Department', 'Gender'])['Admitted'].mean().unstack()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Overall Admission Rate', 'Admission Rate by Department')
)

# 전체 합격률
fig.add_trace(
    go.Bar(x=['Male', 'Female'],
           y=[overall['Male'], overall['Female']],
           marker_color=['#45B7D1', '#FF6B6B'],
           text=[f"{overall['Male']:.0%}", f"{overall['Female']:.0%}"],
           textposition='auto',
           name='Overall'),
    row=1, col=1
)

# 학과별 합격률
for dept in ['Dept A', 'Dept B']:
    color = '#96CEB4' if dept == 'Dept A' else '#FFEAA7'
    fig.add_trace(
        go.Bar(
            x=['Male', 'Female'],
            y=[by_dept.loc[dept, 'Male'], by_dept.loc[dept, 'Female']],
            name=dept,
            marker_color=color,
            text=[f"{by_dept.loc[dept, 'Male']:.0%}",
                  f"{by_dept.loc[dept, 'Female']:.0%}"],
            textposition='auto'
        ),
        row=1, col=2
    )

fig.update_layout(
    title_text="Simpson's Paradox: University Admission Example",
    height=400,
    barmode='group'
)
fig.update_yaxes(title_text='Admission Rate', tickformat='.0%')

fig.show()

print("💡 Simpson's Paradox:")
print(f"  - 전체: 남성 {overall['Male']:.0%} > 여성 {overall['Female']:.0%} → 남성 우대?")
print(f"  - Dept A: 남성 {by_dept.loc['Dept A', 'Male']:.0%} < 여성 {by_dept.loc['Dept A', 'Female']:.0%}")
print(f"  - Dept B: 남성 {by_dept.loc['Dept B', 'Male']:.0%} < 여성 {by_dept.loc['Dept B', 'Female']:.0%}")
print("  → 각 학과에서는 여성 합격률이 더 높다!")
print("  → 여성이 경쟁률 높은 Dept A에 집중 지원했기 때문 (학과 = 교란변수)")

### Simpson's Paradox

전체 데이터에서 관찰되는 상관관계가 **하위 그룹에서는 역전**되는 현상.

**교훈:**
1. 집계된 데이터만 보면 잘못된 결론에 도달할 수 있다
2. 어떤 변수를 통제할지는 **인과 구조에 대한 이해**가 필요하다
3. 데이터 분석만으로는 인과관계를 알 수 없으며, **도메인 지식**이 필수

### 교란변수 식별 체크리스트

| 질문 | 설명 |
|------|------|
| 시간적 선행성 | 처치 이전에 존재하며, 처치와 결과 모두에 영향을 미치는 변수는? |
| 선택 메커니즘 | 처치를 받는 사람/조직은 어떻게 결정되는가? |
| 대안적 설명 | 관찰된 관계를 설명할 수 있는 제3의 요인은? |
| 문헌 검토 | 선행 연구에서 알려진 교란변수는? |
| 전문가 의견 | 전문가들이 중요하다고 생각하는 배경 변수는? |

---

### 📝 이론 과제 5-1 (15분)

#### 과제: 상관관계와 인과관계의 구분

**질문:**

1. **"Correlation does not imply causation"**이라는 표현의 의미를 자신의 말로 설명하고, 비즈니스에서 이 원칙을 무시했을 때 발생할 수 있는 구체적 사례를 하나 제시하시오. (3-4문장)

2. 다음 상황에서 **교란변수**가 무엇인지 식별하시오:
   - "광고비를 많이 쓴 달에 매출이 높았다. 따라서 광고비를 늘리면 매출이 올라간다."
   - 교란변수 식별 체크리스트의 "선택 메커니즘"과 "대안적 설명" 항목을 적용하여 분석하시오.

3. **Simpson's Paradox**를 자신의 말로 설명하고, 이 역설이 기획자에게 주는 교훈을 서술하시오. (2-3문장)

**조사 키워드:**
- "spurious correlation examples"
- "Simpson's paradox Berkeley admission"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

---

### ✅ 과제 5-1 제출란

**학번:** ___________  
**이름:** ___________

#### 1. "Correlation ≠ Causation" 설명 + 사례

(여기에 작성)

#### 2. 교란변수 식별

- 교란변수: 
- 선택 메커니즘 분석: 
- 대안적 설명: 

#### 3. Simpson's Paradox와 교훈

(여기에 작성)

**출처:** (URL)

---

---

## Part 2. 인과 다이어그램 DAG

---

## 5.2 인과 다이어그램 (DAG)

### 기본 개념

**DAG** = **D**irected **A**cyclic **G**raph (방향성 비순환 그래프)

- Judea Pearl이 체계화한 인과구조 시각화 도구
- **노드(Node)**: 변수를 나타냄
- **엣지(Edge)**: 화살표로 인과 방향을 표현 (X → Y = "X가 Y의 원인")
- **방향성**: 인과의 방향을 명시
- **비순환**: 순환 고리가 없음 (시간적 선후관계 반영)

### 3가지 핵심 패턴

| 패턴 | DAG | 통제 필요 | 통제 시 결과 |
|------|-----|----------|------------|
| **교란변수 (Confounder)** | X ← Z → Y | ✅ 필수 | 편향 제거 |
| **매개변수 (Mediator)** | X → M → Y | 상황에 따라 | 간접 효과 제거 |
| **충돌변수 (Collider)** | X → Z ← Y | ❌ 금지 | 허위 상관 발생 |

---

In [ ]:
# DAG 패턴 시각화 함수
def draw_dag(nodes, edges, title, annotations=None):
    """
    간단한 DAG를 Plotly로 시각화한다.
    
    Args:
        nodes: list of dict {'x', 'y', 'label', 'color'}
        edges: list of tuples (from_idx, to_idx)
        title: 그래프 제목
        annotations: 추가 주석 list
    """
    fig = go.Figure()
    
    # 노드 그리기
    for node in nodes:
        fig.add_trace(go.Scatter(
            x=[node['x']], y=[node['y']],
            mode='markers+text',
            marker=dict(size=55, color=node['color']),
            text=[node['label']],
            textposition='middle center',
            textfont=dict(size=10, color='white'),
            showlegend=False
        ))
    
    # 엣지 (화살표)
    for frm, to in edges:
        fig.add_annotation(
            x=nodes[to]['x'], y=nodes[to]['y'],
            ax=nodes[frm]['x'], ay=nodes[frm]['y'],
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=1.5,
            arrowcolor='gray', arrowwidth=2
        )
    
    if annotations:
        for ann in annotations:
            fig.add_annotation(**ann)
    
    fig.update_layout(
        title=title, height=350, width=500,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
                  scaleanchor='x')
    )
    return fig

print("✅ DAG 시각화 함수 정의 완료!")

In [ ]:
# 패턴 1: 교란변수 (Confounder)
fig = draw_dag(
    nodes=[
        {'x': 2, 'y': 3, 'label': 'Z\nConfounder', 'color': '#FF6B6B'},
        {'x': 1, 'y': 1, 'label': 'X\nTreatment', 'color': '#4ECDC4'},
        {'x': 3, 'y': 1, 'label': 'Y\nOutcome', 'color': '#45B7D1'}
    ],
    edges=[(0, 1), (0, 2), (1, 2)],
    title='Pattern 1: Confounder (Must Control ✅)',
    annotations=[
        dict(x=2, y=0.3, text='e.g., Customer Value → Campaign + Purchase',
             showarrow=False, font=dict(size=9, color='gray'))
    ]
)
fig.show()

print("💡 교란변수(Confounder): X ← Z → Y")
print("  - Z를 통제하지 않으면 X→Y 효과가 편향됨")
print("  - 예: 고객 구매성향(Z)이 캠페인 노출(X)과 구매(Y) 모두에 영향")
print("  - 해결: Z를 통제 (층화, 매칭, 회귀 조정)")

In [ ]:
# 패턴 2: 매개변수 (Mediator)
fig = draw_dag(
    nodes=[
        {'x': 1, 'y': 2, 'label': 'X\nEducation', 'color': '#4ECDC4'},
        {'x': 2.5, 'y': 3, 'label': 'M\nSkills', 'color': '#FFEAA7'},
        {'x': 4, 'y': 2, 'label': 'Y\nIncome', 'color': '#45B7D1'}
    ],
    edges=[(0, 1), (1, 2), (0, 2)],
    title='Pattern 2: Mediator (Control Depends on Goal)',
    annotations=[
        dict(x=2.5, y=1.2, text='Direct effect: X→Y (signaling)\n'
             'Indirect effect: X→M→Y (skill building)',
             showarrow=False, font=dict(size=9, color='gray'))
    ]
)
fig.show()

print("💡 매개변수(Mediator): X → M → Y")
print("  - 총 효과를 알려면 M을 통제하지 않는다")
print("  - 직접 효과만 알려면 M을 통제한다")
print("  - 주의: M을 잘못 통제하면 총 효과를 과소추정!")

In [ ]:
# 패턴 3: 충돌변수 (Collider)
fig = draw_dag(
    nodes=[
        {'x': 1, 'y': 3, 'label': 'X\nTalent', 'color': '#4ECDC4'},
        {'x': 3, 'y': 3, 'label': 'Y\nEffort', 'color': '#45B7D1'},
        {'x': 2, 'y': 1, 'label': 'Z\nSuccess\n(Collider)', 'color': '#FF6B6B'}
    ],
    edges=[(0, 2), (1, 2)],
    title='Pattern 3: Collider (Never Control ❌)',
    annotations=[
        dict(x=2, y=3.8, text='X and Y: originally INDEPENDENT',
             showarrow=False, font=dict(size=10, color='red')),
        dict(x=2, y=0.3, text='Controlling Z creates false X↔Y correlation!',
             showarrow=False, font=dict(size=9, color='gray'))
    ]
)
fig.show()

print("💡 충돌변수(Collider): X → Z ← Y")
print("  - X와 Y는 원래 독립 (재능과 노력은 관련 없음)")
print("  - Z를 통제하면(성공한 사람만 분석) X-Y에 허위 상관 발생")
print("  - 재능↑ → 덜 노력해도 성공, 노력↑ → 덜 재능있어도 성공")
print("  - 이것이 '선택 편향(Selection Bias)'")

In [ ]:
# 충돌변수 시뮬레이션: 재능과 노력의 역설
np.random.seed(42)
n = 1000

talent = np.random.normal(50, 15, n)      # 재능 (독립)
effort = np.random.normal(50, 15, n)      # 노력 (독립)
success = talent + effort + np.random.normal(0, 10, n)  # 성공 = 재능 + 노력

# 전체 데이터: 재능과 노력은 독립
corr_all = np.corrcoef(talent, effort)[0, 1]

# 성공한 사람만 (상위 25%) → 충돌변수 통제
success_threshold = np.percentile(success, 75)
mask = success >= success_threshold
corr_success = np.corrcoef(talent[mask], effort[mask])[0, 1]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'All People (r = {corr_all:.2f})',
        f'Successful People Only (r = {corr_success:.2f})'
    )
)

# 전체
fig.add_trace(
    go.Scatter(x=talent, y=effort, mode='markers',
              marker=dict(size=4, color='#4ECDC4', opacity=0.5),
              name='All'),
    row=1, col=1
)

# 성공한 사람만
fig.add_trace(
    go.Scatter(x=talent[mask], y=effort[mask], mode='markers',
              marker=dict(size=5, color='#FF6B6B', opacity=0.7),
              name='Successful'),
    row=1, col=2
)

fig.update_xaxes(title_text='Talent')
fig.update_yaxes(title_text='Effort')
fig.update_layout(
    title_text='Collider Bias: Talent vs Effort',
    height=400, showlegend=False
)

fig.show()

print(f"💡 전체: 재능과 노력의 상관 r = {corr_all:.2f} (거의 독립)")
print(f"💡 성공한 사람만: r = {corr_success:.2f} (부적 상관 → 허위!)")
print("   → 성공(충돌변수)을 통제하면 재능과 노력이 반비례하는 것처럼 보인다")
print("   → 생존 기업만 분석, 응답자만 분석 등에서도 같은 문제 발생")

### 📝 이론 과제 5-2 (15분)

#### 과제: DAG와 변수 유형 구분

**질문:**

1. **Judea Pearl**의 인과 다이어그램(DAG)에서 "방향성(Directed)"과 "비순환성(Acyclic)"이 각각 무엇을 의미하는지 설명하시오.

2. 다음 상황에서 변수의 역할(교란/매개/충돌)을 판단하시오:
   - "디지털 투자(X) → 프로세스 효율(M) → 비용절감(Y)" 에서 M은?
   - "기업규모(Z) → 디지털 투자(X), 기업규모(Z) → 매출(Y)" 에서 Z는?
   - "능력(X) → 채용(Z) ← 학벌(Y)" 에서 Z는?

3. **충돌변수를 통제하면 안 되는 이유**를 "생존 편향(Survivorship Bias)"의 예시를 들어 설명하시오. (2-3문장)

**조사 키워드:**
- "Judea Pearl DAG causal inference"
- "survivorship bias examples"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

---

### ✅ 과제 5-2 제출란

**학번:** ___________  
**이름:** ___________

#### 1. DAG의 방향성과 비순환성

- 방향성: 
- 비순환성: 

#### 2. 변수 유형 판단

- 프로세스 효율(M): ( ) 변수
- 기업규모(Z): ( ) 변수
- 채용(Z): ( ) 변수

#### 3. 충돌변수 통제 금지 + 생존 편향

(여기에 작성)

**출처:** (URL)

---

---

### ☕ 휴식 (15분)

---

---

## Part 3. 인과 효과 추정

---

## 5.3 관찰(Seeing) vs 개입(Doing)

### do-연산자 (Judea Pearl)

| 구분 | 관찰 | 개입 |
|------|------|------|
| 표기 | P(Y \| X=x) | P(Y \| **do**(X=x)) |
| 의미 | X가 x인 것을 **관찰**했을 때 Y | X를 x로 **설정**했을 때 Y |
| 예시 | 약 복용자의 회복률 | 약을 **복용시켰을 때** 회복률 |
| DAG 조작 | 변화 없음 | X로 들어오는 화살표 **제거** |

```
원래 DAG:           개입 후 DAG (do(X=x)):
Z → X → Y           Z   X=x → Y
Z → Y               Z → Y

do(X=x)는 Z → X 화살표를 제거
→ 교란변수 Z의 영향을 차단
```

### 백도어 조정 (Backdoor Adjustment)

관찰 데이터에서 P(Y|do(X=x))를 계산하는 공식:

**P(Y|do(X=x)) = Σ_z P(Y|X=x, Z=z) × P(Z=z)**

→ 교란변수 Z에 대해 조건화하고 가중 평균을 취하면, 관찰 데이터에서 인과 효과 추정 가능!

---

In [ ]:
# 시각화: See vs Do
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Seeing: P(Y | X=x)', 'Doing: P(Y | do(X=x))')
)

# See: 원래 DAG
see_nodes = [
    {'x': 2, 'y': 3, 'label': 'Z'},
    {'x': 1, 'y': 1, 'label': 'X'},
    {'x': 3, 'y': 1, 'label': 'Y'}
]
for n in see_nodes:
    fig.add_trace(go.Scatter(
        x=[n['x']], y=[n['y']], mode='markers+text',
        marker=dict(size=40, color='#4ECDC4'),
        text=[n['label']], textposition='middle center',
        textfont=dict(size=14, color='white'), showlegend=False
    ), row=1, col=1)

# See arrows: Z→X, Z→Y, X→Y (all active)
see_edges = [(2, 3, 1, 1.3), (2, 2.7, 3, 1.3), (1.3, 1, 2.7, 1)]
for ax, ay, x, y in see_edges:
    fig.add_annotation(x=x, y=y, ax=ax, ay=ay,
                       xref='x', yref='y', axref='x', ayref='y',
                       showarrow=True, arrowhead=2, arrowcolor='gray', arrowwidth=2)

# Do: 개입 DAG
do_nodes = [
    {'x': 2, 'y': 3, 'label': 'Z', 'color': '#4ECDC4'},
    {'x': 1, 'y': 1, 'label': 'X=x', 'color': '#FF6B6B'},
    {'x': 3, 'y': 1, 'label': 'Y', 'color': '#4ECDC4'}
]
for n in do_nodes:
    fig.add_trace(go.Scatter(
        x=[n['x']], y=[n['y']], mode='markers+text',
        marker=dict(size=40, color=n.get('color', '#4ECDC4')),
        text=[n['label']], textposition='middle center',
        textfont=dict(size=14, color='white'), showlegend=False
    ), row=1, col=2)

# Do arrows: Z→Y, X→Y (Z→X removed!)
do_edges = [(2, 2.7, 3, 1.3), (1.3, 1, 2.7, 1)]
for ax, ay, x, y in do_edges:
    fig.add_annotation(x=x, y=y, ax=ax, ay=ay,
                       xref='x2', yref='y2', axref='x2', ayref='y2',
                       showarrow=True, arrowhead=2, arrowcolor='gray', arrowwidth=2)

# X marking removed arrow
fig.add_annotation(x=1.5, y=2.2, text='❌ Removed', xref='x2', yref='y2',
                   showarrow=False, font=dict(size=11, color='red'))

for col in [1, 2]:
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=col)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=col)

fig.update_layout(title_text='See vs Do: The Core of Causal Inference', height=350)
fig.show()

print("💡 See: 데이터에서 X=x인 경우를 관찰 (교란 포함)")
print("💡 Do:  X를 x로 강제 설정 (Z→X 차단 → 교란 제거)")
print("   → 무작위 실험(RCT)은 do()를 물리적으로 구현한 것")
print("   → 실험 불가능 시, 백도어 조정으로 관찰 데이터에서 do() 계산")

## 5.4 반사실적 사고 (Counterfactual)

### 핵심 질문

> "만약 다르게 했다면 어땠을까?"

### 개인 처치 효과

개인 i의 처치 효과 = Y₁ᵢ - Y₀ᵢ

- Y₁ᵢ: 처치를 받았을 때의 결과
- Y₀ᵢ: 처치를 받지 않았을 때의 결과
- **문제**: 같은 사람이 동시에 처치를 받고 안 받을 수 없음 → 인과추론의 근본적 문제

### 평균 처치 효과 (ATE)

**ATE** = E[Y₁] - E[Y₀]

- 무작위 실험(RCT)에서는 처치/통제 그룹이 동질 → 단순 비교로 ATE 추정
- 관찰 데이터에서는 선택 편향 → 교란변수 통제 필요

## 5.5 준실험 설계

무작위 실험이 불가능할 때 관찰 데이터에서 인과 효과를 추정하는 방법:

| 방법 | 핵심 가정 | 적용 상황 | 예시 |
|------|----------|----------|------|
| **이중차분법 (DID)** | 평행 추세 | 정책 전후 비교 | 최저임금 인상 → 고용 효과 |
| **회귀단절설계 (RDD)** | 기준점 주변 동질성 | 기준점 기반 처치 | 장학금 기준 80점 → 학업 효과 |
| **도구변수법 (IV)** | 도구변수 유효성 | 내생성 존재 시 | 군 복무(징병 추첨) → 소득 효과 |

---

In [ ]:
# 이중차분법 (DID) 시뮬레이션
np.random.seed(42)

time = np.array([0, 1, 2, 3, 4, 5, 6, 7])
treatment_time = 4  # 정책 시행 시점

# 처치 그룹 (정책 대상)
treated_before = 20 + 2 * time[:treatment_time] + np.random.normal(0, 0.5, treatment_time)
treated_after = treated_before[-1] + 5 + 2 * np.arange(len(time[treatment_time:])) + np.random.normal(0, 0.5, len(time[treatment_time:]))
treated = np.concatenate([treated_before, treated_after])

# 통제 그룹 (정책 미대상)
control = 18 + 2 * time + np.random.normal(0, 0.5, len(time))

# 반사실 (정책 없었을 경우의 처치 그룹)
counterfactual = 20 + 2 * time + np.random.normal(0, 0.3, len(time))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=time, y=treated, mode='lines+markers',
    name='Treatment Group (Observed)',
    line=dict(color='#FF6B6B', width=3)
))

fig.add_trace(go.Scatter(
    x=time, y=control, mode='lines+markers',
    name='Control Group',
    line=dict(color='#4ECDC4', width=3)
))

fig.add_trace(go.Scatter(
    x=time[treatment_time:], y=counterfactual[treatment_time:],
    mode='lines', name='Counterfactual (No Policy)',
    line=dict(color='#FF6B6B', width=2, dash='dash')
))

# 정책 시행 시점
fig.add_vline(x=treatment_time - 0.5, line_dash='dash', line_color='gray',
              annotation_text='Policy Implementation')

# 인과 효과 표시
fig.add_annotation(
    x=6, y=(treated[6] + counterfactual[6]) / 2,
    text=f'Causal Effect\n(DID = {treated[6] - counterfactual[6]:.1f})',
    showarrow=True, arrowhead=2, ax=50, ay=0,
    font=dict(color='red', size=12)
)

fig.update_layout(
    title='Difference-in-Differences (DID): Policy Effect Estimation',
    xaxis_title='Time Period',
    yaxis_title='Outcome',
    height=450
)

fig.show()

did_effect = (treated[-1] - treated[treatment_time-1]) - (control[-1] - control[treatment_time-1])
print(f"💡 DID 추정량 = (처치 후-전) - (통제 후-전) = {did_effect:.1f}")
print("  = 처치그룹 변화에서 자연적 추세(통제그룹)를 빼면 순수 정책 효과")
print("  핵심 가정: 정책 없었으면 두 그룹이 같은 추세를 보였을 것 (평행 추세)")

### 📝 이론 과제 5-3 (10분)

#### 과제: 인과 효과 추정

**질문:**

1. **do-연산자**의 핵심 아이디어를 "무작위 실험(RCT)"과 연결하여 설명하시오. 왜 RCT가 인과 효과 추정의 "gold standard"인지 서술하시오. (2-3문장)

2. **이중차분법(DID)**의 핵심 가정인 "평행 추세"가 무엇인지 설명하고, 이 가정이 위배되는 상황의 예를 들어 설명하시오.

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

---

### ✅ 과제 5-3 제출란

**학번:** ___________  
**이름:** ___________

#### 1. do-연산자와 RCT

(여기에 작성)

#### 2. 이중차분법의 평행 추세 가정

- 평행 추세란: 
- 위배되는 상황 예: 

---

---

## Part 4. 실습: 마케팅 캠페인 효과 시뮬레이션

---

### 실습 개요

Python으로 **교란변수가 있는 마케팅 데이터**를 시뮬레이션하고,  
단순 비교 vs 교란 통제 비교를 통해 인과 효과 추정의 중요성을 확인한다.

1. 교란변수가 있는 데이터 생성
2. 단순 비교 (편향된 추정)
3. 교란변수 통제 후 비교 (정확한 추정)
4. 결과 비교 시각화

---

In [ ]:
# 마케팅 캠페인 데이터 시뮬레이션 함수

def generate_campaign_data(n=5000, true_effect=0.15, seed=42):
    """
    교란변수가 있는 마케팅 캠페인 데이터를 생성한다.
    
    인과 구조:
    - customer_value(Z) → campaign(X) : 구매성향 높은 고객에게 더 많이 노출
    - customer_value(Z) → purchase(Y) : 구매성향 높은 고객이 더 많이 구매
    - campaign(X) → purchase(Y) : 캠페인의 순수 인과 효과 (추정 대상)
    
    Args:
        n: 고객 수
        true_effect: 실제 인과 효과 (구매 확률 증가분)
        seed: 랜덤 시드
    
    Returns:
        DataFrame, true_effect
    """
    np.random.seed(seed)
    
    # 교란변수: 고객 구매성향 (0~1)
    customer_value = np.random.beta(2, 5, n)
    
    # 공변량
    recency = np.random.exponential(30, n)  # 마지막 구매 후 일수
    frequency = np.random.poisson(3, n)     # 과거 구매 횟수
    
    # 처치: 캠페인 노출 (교란변수의 영향을 받음)
    campaign_prob = 0.3 + 0.4 * customer_value - 0.005 * recency + 0.05 * frequency
    campaign_prob = np.clip(campaign_prob, 0.05, 0.95)
    campaign = np.random.binomial(1, campaign_prob)
    
    # 결과: 구매 여부 (처치 + 교란변수 모두 영향)
    purchase_prob = (
        0.1                          # 기본 구매율
        + 0.5 * customer_value       # 교란: 구매성향
        + true_effect * campaign     # 인과 효과: 캠페인
        - 0.002 * recency            # recency 영향
        + 0.02 * frequency           # frequency 영향
        + np.random.normal(0, 0.1, n)
    )
    purchase_prob = np.clip(purchase_prob, 0, 1)
    purchase = np.random.binomial(1, purchase_prob)
    
    df = pd.DataFrame({
        'customer_value': customer_value,
        'recency': recency,
        'frequency': frequency,
        'campaign': campaign,
        'purchase': purchase
    })
    
    return df, true_effect

# 데이터 생성
df, TRUE_EFFECT = generate_campaign_data(n=5000, true_effect=0.15)

print("✅ 마케팅 캠페인 데이터 생성 완료!")
print(f"  - 고객 수: {len(df):,}명")
print(f"  - 캠페인 노출률: {df['campaign'].mean():.1%}")
print(f"  - 전체 구매율: {df['purchase'].mean():.1%}")
print(f"  - 실제 인과 효과 (True ATE): {TRUE_EFFECT}")

In [ ]:
# 교란변수 확인: 캠페인 그룹과 비캠페인 그룹의 차이

df_treated = df[df['campaign'] == 1]
df_control = df[df['campaign'] == 0]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Customer Value Distribution',
        'Campaign Rate by Customer Value',
        'Purchase Rate by Customer Value'
    )
)

# 1. 교란변수 분포 비교
for group, name, color in [(df_treated, 'Campaign O', '#FF6B6B'),
                            (df_control, 'Campaign X', '#4ECDC4')]:
    fig.add_trace(
        go.Histogram(x=group['customer_value'], name=name,
                     marker_color=color, opacity=0.6,
                     histnorm='probability density'),
        row=1, col=1
    )

# 2. 고객가치별 캠페인 노출률
df['value_bin'] = pd.cut(df['customer_value'], bins=10)
campaign_rate = df.groupby('value_bin', observed=True)['campaign'].mean()
fig.add_trace(
    go.Bar(x=list(range(len(campaign_rate))), y=campaign_rate.values,
           marker_color='#45B7D1', name='Campaign Rate', showlegend=False),
    row=1, col=2
)

# 3. 고객가치별 구매율
purchase_rate = df.groupby('value_bin', observed=True)['purchase'].mean()
fig.add_trace(
    go.Bar(x=list(range(len(purchase_rate))), y=purchase_rate.values,
           marker_color='#96CEB4', name='Purchase Rate', showlegend=False),
    row=1, col=3
)

fig.update_layout(
    title_text='Confounding Evidence: Customer Value Affects Both Campaign & Purchase',
    height=400, barmode='overlay'
)

fig.show()

print("💡 교란 증거:")
print(f"  - 캠페인 그룹의 평균 구매성향: {df_treated['customer_value'].mean():.3f}")
print(f"  - 비캠페인 그룹의 평균 구매성향: {df_control['customer_value'].mean():.3f}")
print("  → 캠페인 그룹이 원래 구매성향이 더 높다! (교란)")

In [ ]:
# 단순 비교 vs 교란 통제 비교

# 방법 1: 단순 비교 (편향된)
naive_effect = df_treated['purchase'].mean() - df_control['purchase'].mean()

# 방법 2: 교란변수 통제 - 층화 (Stratification)
df['value_quintile'] = pd.qcut(df['customer_value'], q=5, labels=False)

stratum_effects = []
stratum_sizes = []
for q in range(5):
    stratum = df[df['value_quintile'] == q]
    treated_rate = stratum[stratum['campaign'] == 1]['purchase'].mean()
    control_rate = stratum[stratum['campaign'] == 0]['purchase'].mean()
    effect = treated_rate - control_rate
    stratum_effects.append(effect)
    stratum_sizes.append(len(stratum))

# 가중 평균
total_n = sum(stratum_sizes)
stratified_effect = sum(e * s / total_n for e, s in zip(stratum_effects, stratum_sizes))

# 방법 3: 선형 회귀 조정
from numpy.linalg import lstsq
X_reg = np.column_stack([
    np.ones(len(df)),
    df['campaign'].values,
    df['customer_value'].values,
    df['recency'].values,
    df['frequency'].values
])
y_reg = df['purchase'].values
coeffs = lstsq(X_reg, y_reg, rcond=None)[0]
regression_effect = coeffs[1]  # campaign 계수

# 결과 비교
results = pd.DataFrame({
    'Method': ['Naive Comparison', 'Stratification', 'Linear Regression', 'True Effect'],
    'Estimated ATE': [naive_effect, stratified_effect, regression_effect, TRUE_EFFECT],
    'Bias': [naive_effect - TRUE_EFFECT, stratified_effect - TRUE_EFFECT,
             regression_effect - TRUE_EFFECT, 0]
})

# 시각화
colors = ['#FF6B6B', '#FFEAA7', '#4ECDC4', '#45B7D1']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=results['Method'],
    y=results['Estimated ATE'],
    marker_color=colors,
    text=[f"{v:.3f}" for v in results['Estimated ATE']],
    textposition='auto'
))

fig.add_hline(y=TRUE_EFFECT, line_dash='dash', line_color='gray',
              annotation_text=f'True Effect = {TRUE_EFFECT}')

fig.update_layout(
    title='Causal Effect Estimation: Naive vs Controlled',
    yaxis_title='Estimated ATE (Campaign Effect)',
    height=450
)

fig.show()

print("\n📊 추정 결과 비교:")
print("=" * 65)
print(f"{'Method':<25} {'ATE':<12} {'Bias':<12} {'Status'}")
print("-" * 65)
for _, row in results.iterrows():
    status = '✅' if abs(row['Bias']) < 0.03 else '⚠️ Biased'
    print(f"{row['Method']:<25} {row['Estimated ATE']:<12.4f} {row['Bias']:<+12.4f} {status}")
print("=" * 65)
print("\n💡 단순 비교는 교란변수로 인해 캠페인 효과를 과대추정!")
print("   교란변수를 통제하면 실제 효과에 가까운 추정이 가능")

### 실습 과제 5-4: 교란변수 통제 실험

위 시뮬레이션에서 **True Effect를 0.05로 변경**하여 실험하시오.

**요구사항:**
1. `generate_campaign_data(n=5000, true_effect=0.05)`로 데이터 재생성
2. 단순 비교와 층화 비교의 결과 차이를 계산
3. "효과가 작을 때 교란변수 통제가 더 중요한 이유"를 서술

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO: 1. True Effect = 0.05로 데이터 재생성
# df_small, TRUE_SMALL = generate_campaign_data(n=5000, true_effect=0.05)

# TODO: 2. 단순 비교
# naive = df_small[df_small['campaign']==1]['purchase'].mean() - \
#         df_small[df_small['campaign']==0]['purchase'].mean()
# print(f"Naive: {naive:.4f}")

# TODO: 3. 층화 비교 (위 코드 참고하여 작성)

# TODO: 4. 결과 해석
# print("효과가 작을 때 교란변수 통제가 더 중요한 이유:")
# print("...")

# ========== 여기까지 ==========

---

## Part 5. 종합 실습: DAG 구축 + 인과 효과 추정

---

### 실습 시나리오: 디지털 전환 투자 효과 분석

| 항목 | 내용 |
|------|------|
| 문제 | A 그룹사: 디지털 투자가 실제로 경영 성과를 개선하는가? |
| 처치 (X) | 디지털 전환 투자 수준 |
| 결과 (Y) | 매출 성장률, 비용 절감률 |
| 교란변수 후보 | 기업규모, 산업특성, 경영진 역량 |
| 매개변수 후보 | 프로세스 효율, 직원 디지털 역량 |

---

In [ ]:
# 디지털 전환 DAG 시각화

fig = go.Figure()

# 노드 정의
dt_nodes = [
    {'x': 1, 'y': 4, 'label': 'Company\nSize', 'color': '#FF6B6B', 'type': 'Confounder'},
    {'x': 3, 'y': 4, 'label': 'Industry\nType', 'color': '#FF6B6B', 'type': 'Confounder'},
    {'x': 2, 'y': 3, 'label': 'Digital\nInvestment', 'color': '#4ECDC4', 'type': 'Treatment'},
    {'x': 1, 'y': 2, 'label': 'Process\nEfficiency', 'color': '#FFEAA7', 'type': 'Mediator'},
    {'x': 3, 'y': 2, 'label': 'Employee\nCapability', 'color': '#FFEAA7', 'type': 'Mediator'},
    {'x': 1, 'y': 1, 'label': 'Cost\nReduction', 'color': '#45B7D1', 'type': 'Outcome'},
    {'x': 3, 'y': 1, 'label': 'Revenue\nGrowth', 'color': '#45B7D1', 'type': 'Outcome'}
]

for node in dt_nodes:
    fig.add_trace(go.Scatter(
        x=[node['x']], y=[node['y']], mode='markers+text',
        marker=dict(size=55, color=node['color']),
        text=[node['label']], textposition='middle center',
        textfont=dict(size=9, color='white'),
        name=f"{node['label'].replace(chr(10), ' ')} ({node['type']})",
        hovertemplate=f"<b>{node['label'].replace(chr(10), ' ')}</b><br>Type: {node['type']}<extra></extra>"
    ))

# 엣지
edges = [
    (0, 2), (1, 2),  # Confounders → Treatment
    (0, 4), (1, 6),  # Confounders → Outcomes (bypass)
    (2, 3), (2, 4),  # Treatment → Mediators
    (3, 5), (4, 6),  # Mediators → Outcomes
    (4, 3)           # Employee → Process
]

for frm, to in edges:
    fig.add_annotation(
        x=dt_nodes[to]['x'], y=dt_nodes[to]['y'],
        ax=dt_nodes[frm]['x'], ay=dt_nodes[frm]['y'],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.2,
        arrowcolor='gray', arrowwidth=1.5
    )

# 범례 추가
fig.add_annotation(x=0.5, y=0.3, text='🔴 Confounder  🟢 Treatment  🟡 Mediator  🔵 Outcome',
                   showarrow=False, font=dict(size=10))

fig.update_layout(
    title='DAG: Digital Transformation Impact on Business Performance',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[0, 4]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[0, 5]),
    height=550, showlegend=False
)

fig.show()

print("💡 분석 전략:")
print("  - 총 효과: 교란변수만 통제 (기업규모, 산업), 매개변수는 통제하지 않음")
print("  - 직접 효과: 매개변수도 통제 → 프로세스 효율화를 거치지 않는 효과")
print("  - 간접 효과: 총 효과 - 직접 효과 → 프로세스를 통한 효과")

In [ ]:
# 디지털 전환 효과 분해 시뮬레이션
np.random.seed(42)
n_firms = 200

# 교란변수
firm_size = np.random.lognormal(5, 1, n_firms)  # 기업규모 (매출)
industry_tech = np.random.binomial(1, 0.4, n_firms)  # IT산업 여부

# 처치: 디지털 투자 (교란변수 영향 받음)
digital_invest = (
    2 + 0.001 * firm_size + 3 * industry_tech
    + np.random.normal(0, 2, n_firms)
)
digital_invest = np.clip(digital_invest, 0, 15)

# 매개변수: 프로세스 효율
process_eff = 0.5 * digital_invest + np.random.normal(0, 1, n_firms)

# 결과: ROE 개선 (%p)
TRUE_TOTAL = 0.25   # 디지털 투자 1단위 → ROE +0.25%p (총 효과)
TRUE_DIRECT = 0.08  # 직접 효과
TRUE_INDIRECT = 0.17  # 간접 효과 (프로세스를 통해)

roe_improvement = (
    TRUE_DIRECT * digital_invest      # 직접 효과
    + 0.34 * process_eff              # 간접 효과 (매개)
    + 0.0005 * firm_size              # 교란
    + 2 * industry_tech               # 교란
    + np.random.normal(0, 1.5, n_firms)
)

df_dt = pd.DataFrame({
    'firm_size': firm_size,
    'industry_tech': industry_tech,
    'digital_invest': digital_invest,
    'process_eff': process_eff,
    'roe_improvement': roe_improvement
})

# 방법별 추정
# 1. 단순 상관 (편향)
corr_naive = np.corrcoef(digital_invest, roe_improvement)[0, 1]

# 2. 교란 통제 (총 효과)
X_total = np.column_stack([np.ones(n_firms), digital_invest, firm_size, industry_tech])
coeffs_total = lstsq(X_total, roe_improvement, rcond=None)[0]

# 3. 교란 + 매개 통제 (직접 효과)
X_direct = np.column_stack([np.ones(n_firms), digital_invest, firm_size, industry_tech, process_eff])
coeffs_direct = lstsq(X_direct, roe_improvement, rcond=None)[0]

# 결과 시각화
effect_data = pd.DataFrame({
    'Effect Type': ['Total Effect', 'Direct Effect', 'Indirect Effect\n(via Process)'],
    'Estimated': [coeffs_total[1], coeffs_direct[1], coeffs_total[1] - coeffs_direct[1]],
    'True': [TRUE_TOTAL, TRUE_DIRECT, TRUE_INDIRECT]
})

fig = go.Figure()
fig.add_trace(go.Bar(
    x=effect_data['Effect Type'], y=effect_data['Estimated'],
    name='Estimated', marker_color='#4ECDC4',
    text=[f"{v:.3f}" for v in effect_data['Estimated']],
    textposition='auto'
))
fig.add_trace(go.Bar(
    x=effect_data['Effect Type'], y=effect_data['True'],
    name='True', marker_color='#FF6B6B',
    text=[f"{v:.3f}" for v in effect_data['True']],
    textposition='auto'
))

fig.update_layout(
    title='Digital Transformation: Effect Decomposition',
    yaxis_title='ROE Improvement (%p per unit investment)',
    barmode='group', height=450
)

fig.show()

indirect_pct = (coeffs_total[1] - coeffs_direct[1]) / coeffs_total[1] * 100
print(f"💡 디지털 전환 효과 분해 결과:")
print(f"  - 총 효과: ROE +{coeffs_total[1]:.3f}%p (투자 1단위당)")
print(f"  - 직접 효과: ROE +{coeffs_direct[1]:.3f}%p")
print(f"  - 간접 효과: ROE +{coeffs_total[1] - coeffs_direct[1]:.3f}%p (프로세스 효율화 경유)")
print(f"  → 총 효과의 약 {indirect_pct:.0f}%가 프로세스 효율화를 통해 발생")
print(f"  → 시사점: 디지털 투자 + 프로세스 재설계가 동반되어야 효과 극대화")

### 실습 과제 5-5: 나만의 DAG 구축과 분석 전략

다음 시나리오 중 하나를 선택하여, 인과 다이어그램(DAG)을 구축하고 분석 전략을 제시하시오.

**시나리오 A:** 온라인 교육 프로그램이 학업 성취에 미치는 효과  
**시나리오 B:** SNS 마케팅이 브랜드 인지도와 매출에 미치는 효과  
**시나리오 C:** 재택근무 정책이 직원 생산성에 미치는 효과

**작성 내용:**
1. 변수 목록 (처치, 결과, 교란변수, 매개변수)
2. DAG 구조 (텍스트로 화살표 표현)
3. 분석 전략 (어떤 변수를 통제할 것인가? 왜?)

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO: 1. 선택한 시나리오와 변수 목록
print("시나리오: ___")
print("")
print("변수 목록:")
print("  처치(X): ")
print("  결과(Y): ")
print("  교란변수(Z): ")
print("  매개변수(M): ")

# TODO: 2. DAG 구조
print("\nDAG 구조:")
print("  Z → X → M → Y")
print("  Z → Y")
# (자유롭게 수정)

# TODO: 3. 분석 전략
print("\n분석 전략:")
print("  총 효과 추정: 통제할 변수 = ")
print("  직접 효과 추정: 통제할 변수 = ")
print("  이유: ")

# TODO: 4. (선택) draw_dag 함수로 시각화

# ========== 여기까지 ==========

---

## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

1. **상관관계 ≠ 인과관계**: 교란변수에 의한 허위 상관, Simpson's Paradox에 주의
2. **DAG로 인과구조 시각화**: 교란변수(통제 필수), 매개변수(상황에 따라), 충돌변수(통제 금지)
3. **See vs Do**: 관찰과 개입은 다르다 → do-연산자로 인과 효과 정의
4. **준실험 설계**: DID, RDD, IV로 관찰 데이터에서도 인과 추정 가능
5. **교란변수 통제의 위력**: 단순 비교 vs 통제 비교의 극적 차이를 시뮬레이션으로 확인

### 핵심 개념 요약

| 개념 | 핵심 원칙 | 실무 적용 |
|------|----------|----------|
| 상관 ≠ 인과 | 함께 움직임 ≠ 유발 | 정책 설계 시 인과관계 확인 필수 |
| 교란변수 | 처치와 결과 모두에 영향 | DAG로 식별 → 통제 |
| 충돌변수 | 통제하면 허위 상관 발생 | 생존 편향, 선택 편향 주의 |
| do-연산자 | 개입의 효과를 정의 | RCT 또는 백도어 조정 |
| 효과 분해 | 총 = 직접 + 간접 | 매개경로의 중요성 파악 |

### 핵심 메시지

> 기획은 **"무엇을 하면 어떤 결과가 나올까?"**에 답하는 것이다.  
> 이 질문에 제대로 답하려면 상관관계가 아닌 **인과관계**를 이해해야 한다.  
> 인과추론 없는 기획은 잘못된 방향으로 자원을 낭비할 위험이 있다.

### 3-4장과의 연결

- **3장(로직 트리)**: Why Tree에서 도출한 원인 가설 → 5장에서 인과추론으로 **검증**
- **4장(이슈 정의)**: "디지털 역량 부족이 매출 정체의 원인인가?" → 상관? 인과? 검증 필요
- **DAG**: 로직 트리의 가정(A→B→C)을 **인과 구조로 명시화**

### 다음 장 예고

**제6장: 시스템 다이내믹스**
- 5장의 **선형적** 인과관계 → 6장의 **피드백 루프**를 포함한 동적 시스템으로 확장
- 캠페인 → 구매 → 재구매 → 캠페인 노출 증가 (순환적 관계)
- DAG는 비순환이지만, 현실의 많은 시스템은 순환적

### 📚 추가 학습 자료

- **Judea Pearl & Dana Mackenzie** (2018). *The Book of Why*. Basic Books.
- **Miguel Hernán & James Robins** (2020). *Causal Inference: What If*. CRC Press.
- **Scott Cunningham** (2021). *Causal Inference: The Mixtape*. Yale University Press.
- Python: `dowhy` (인과추론), `econml` (인과 머신러닝), `causal-learn` (인과 발견)

---

**제5장 끝**